# 001 — Embed both corpora (US + BR)

Single modular workflow. Embeds the sentence-chunk corpora with the **BERTopic-standard** sentence transformers:

| corpus | model | dim | output |
|---|---|---|---|
| US (English) | `all-MiniLM-L6-v2` | 384 | `data/us/emb_minilm.npy` |
| BR (multilingual) | `paraphrase-multilingual-MiniLM-L12-v2` | 384 | `data/br/emb_minilm_multi.npy` |

Each `emb_*.npy` is paired with `*_chunk_ids.npy` (the `chunk_id` join key, same row order).

In [2]:
import os, gc, time
from datetime import datetime
import numpy as np
import pyarrow.feather as feather
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from google.colab import drive
drive.mount('/content/drive')

ROOT = os.getenv('TOPIC2IRT_ROOT', '/content/drive/MyDrive/Papers/transfer_learning/topic2irt')

CORPORA = {
    'us': {
        'feather': f'{ROOT}/data/us/campaignview_chunks_sent.feather',
        'model':   'all-MiniLM-L6-v2',
        'emb':     f'{ROOT}/data/us/emb_minilm.npy',
        'ids':     f'{ROOT}/data/us/emb_minilm_chunk_ids.npy',
    },
    'br': {
        'feather': f'{ROOT}/data/br/br_manifestos_chunks_sent.feather',
        'model':   'paraphrase-multilingual-MiniLM-L12-v2',
        'emb':     f'{ROOT}/data/br/emb_minilm_multi.npy',
        'ids':     f'{ROOT}/data/br/emb_minilm_multi_chunk_ids.npy',
    },
}

# Wall time of each step, appended to the pipeline register at the end of the notebook.
RUN_STARTED = datetime.now()
TIMING = {'us': {}, 'br': {}}
SCALE  = {'us': {}, 'br': {}}

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
for name, cfg in CORPORA.items():
    print(name, '->', os.path.basename(cfg['feather']), '|', cfg['model'],
          '| exists:', os.path.exists(cfg['feather']))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
us -> campaignview_chunks_sent.feather | all-MiniLM-L6-v2 | exists: True
br -> br_manifestos_chunks_sent.feather | paraphrase-multilingual-MiniLM-L12-v2 | exists: True


In [5]:
import shutil

CHUNK_SIZE = 100_000   # rows per checkpoint block
BATCH      = 512       # encode batch (T4-safe for both MiniLM models)

def embed_corpus(name, cfg):
    if os.path.exists(cfg['emb']):
        print(f"[{name}] {np.load(cfg['emb'], mmap_mode='r').shape} already on disk -> skip")
        return

    _t0 = time.perf_counter()
    df    = feather.read_table(cfg['feather'], columns=['chunk_id', 'chunk_text']).to_pandas()
    # --- length-sort ONLY for encoding throughput (~1.5x); we UNSORT before saving so the ---
    # --- saved emb + chunk_ids are in the SAME row order as the feather (alignment guarantee). ---
    order = np.argsort(df['chunk_text'].str.len().to_numpy(), kind='stable')
    texts = df['chunk_text'].to_numpy()[order].tolist()
    n     = len(texts)
    TIMING[name]['1 load+sort'] = time.perf_counter() - _t0
    SCALE[name] = {'n_docs': '', 'n_chunks': n}
    print(f"[{name}] {n:,} chunks | {cfg['model']} | length-sorted for encode, saved in feather order")

    _t0 = time.perf_counter()
    model = SentenceTransformer(cfg['model'], device='cuda')
    pdir  = cfg['emb'].replace('.npy', '_parts')
    os.makedirs(pdir, exist_ok=True)

    # --- encode in blocks (resume: skip blocks already on disk) ---
    for i in tqdm(range(0, n, CHUNK_SIZE), unit='blk', desc=f'{name} encode'):
        bf = f"{pdir}/{i:09d}.npy"
        if os.path.exists(bf):
            continue
        emb = model.encode(texts[i:i+CHUNK_SIZE], batch_size=BATCH,
                           show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
        np.save(bf, emb)
    del model; gc.collect(); torch.cuda.empty_cache()
    TIMING[name]['2 encode'] = time.perf_counter() - _t0

    # --- stream blocks into a disk-backed memmap, SCATTERING each row back to its ORIGINAL ---
    # --- feather position via `order` (so out[k] is the embedding of df.iloc[k]). ---
    # The scatter is random-access, which crawls on the Drive FUSE mount (US: 15:50 for
    # 675 MB), so the memmap is built on the LOCAL disk and copied to Drive in one
    # sequential pass at the end.  The bytes that land on Drive are identical.
    _t0 = time.perf_counter()
    dim = np.load(f"{pdir}/000000000.npy", mmap_mode='r').shape[1]
    tmp = f"/content/{os.path.basename(cfg['emb'])}"
    out = np.lib.format.open_memmap(tmp, mode='w+', dtype=np.float32, shape=(n, dim))
    pos = 0
    for i in tqdm(range(0, n, CHUNK_SIZE), unit='blk', desc=f'{name} unsort'):
        blk = np.load(f"{pdir}/{i:09d}.npy")
        out[order[pos:pos+len(blk)]] = blk        # UNSORT: scatter to original feather rows
        pos += len(blk)
    out.flush(); del out
    assert pos == n, f"{pos} != {n}"
    TIMING[name]['3 unsort (local)'] = time.perf_counter() - _t0

    _t0 = time.perf_counter()
    shutil.copyfile(tmp, cfg['emb'])              # one sequential write to Drive
    os.remove(tmp)
    np.save(cfg['ids'], df['chunk_id'].to_numpy())   # chunk_ids in ORIGINAL feather order (row-aligned to emb)
    shutil.rmtree(pdir)
    TIMING[name]['4 copy to Drive'] = time.perf_counter() - _t0
    print(f"  saved emb {n:,}x{dim} (feather order) -> {os.path.basename(cfg['emb'])}")

print("embed_corpus defined")


embed_corpus defined


In [4]:
embed_corpus('us', CORPORA['us'])


[us] 439,282 chunks | all-MiniLM-L6-v2 | length-sorted for encode, saved in feather order


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

us write: 100%|██████████| 5/5 [15:50<00:00, 190.05s/blk]


  saved emb 439,282x384 (feather order) -> emb_minilm.npy


In [6]:
embed_corpus('br', CORPORA['br'])


[br] 3,173,387 chunks | paraphrase-multilingual-MiniLM-L12-v2 | length-sorted for encode, saved in feather order


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

br unsort: 100%|██████████| 32/32 [01:31<00:00,  2.87s/blk]


  saved emb 3,173,387x384 (feather order) -> emb_minilm_multi.npy


In [7]:
for name, cfg in CORPORA.items():
    n_feather = feather.read_table(cfg['feather'], columns=['chunk_id']).num_rows
    emb = np.load(cfg['emb'], mmap_mode='r')
    ids = np.load(cfg['ids'], allow_pickle=True)
    parts_left = os.path.exists(cfg['emb'].replace('.npy', '_parts'))
    print(f"[{name}] emb {emb.shape} {emb.dtype} | ids {ids.shape} {ids.dtype} | feather {n_feather:,} "
          f"| match={emb.shape[0]==len(ids)==n_feather} | parts_left={parts_left}")


[us] emb (439282, 384) float32 | ids (439282,) object | feather 439,282 | match=True | parts_left=False
[br] emb (3173387, 384) float32 | ids (3173387,) object | feather 3,173,387 | match=True | parts_left=False


In [ ]:
pdir = CORPORA['br']['emb'].replace('.npy', '_parts')
if os.path.exists(pdir):
    blocks = sorted(os.listdir(pdir))
    print(f"{len(blocks)} block files in {pdir}")
    print("first:", blocks[0], "| last:", blocks[-1])
else:
    print("NO parts dir — blocks gone, BR must re-encode from scratch")


NO parts dir — blocks gone, BR must re-encode from scratch


In [ ]:
# Append this run to the pipeline timing register.  Every timed stage writes the same two
# files under reports/timing/, and code/timing.py is their only writer.
import sys
sys.path.insert(0, f'{ROOT}/code')
from timing import log_run, CSV_PATH, MD_PATH

timing = {c: t for c, t in TIMING.items() if t}          # a skipped corpus contributes nothing
scale  = {c: SCALE[c] for c in timing}
gpu    = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
note   = (f"{gpu}. US all-MiniLM-L6-v2, BR paraphrase-multilingual-MiniLM-L12-v2, "
          f"batch {BATCH}, {CHUNK_SIZE:,}-row checkpoint blocks.")

print(log_run('01 embed', timing, scale=scale, started=RUN_STARTED, note=note))
print('->', CSV_PATH)
print('->', MD_PATH)
